In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load saved model and encoders
model      = joblib.load('models/yield_model.pkl')
le_state   = joblib.load('models/le_state.pkl')
le_district = joblib.load('models/le_district.pkl')
le_season  = joblib.load('models/le_season.pkl')
le_crop    = joblib.load('models/le_crop.pkl')

df = pd.read_csv('../dataset/merged_crop_data.csv')

print("✅ Everything loaded")
print("Known states:", len(le_state.classes_))
print("Known crops:", len(le_crop.classes_))

✅ Everything loaded
Known states: 22
Known crops: 54


In [2]:
price_lookup = df.groupby(['State_Name', 'Crop'])['Avg_Price'].mean().reset_index()
price_lookup.columns = ['State_Name', 'Crop', 'Avg_Price']

viable_crops = df.groupby(['State_Name', 'District_Name', 'Season'])['Crop']\
                 .apply(list).reset_index()
viable_crops.columns = ['State_Name', 'District_Name', 'Season', 'Viable_Crops']

rainfall_lookup = df.groupby(['State_Name', 'District_Name'])['Annual_Rainfall']\
                    .mean().reset_index()

print("✅ Lookup tables built")
print("Price lookup shape:", price_lookup.shape)

✅ Lookup tables built
Price lookup shape: (257, 3)


In [3]:
def recommend_crops(state, district, season, land_area_hectares, top_k=5):
    """
    Given farmer inputs, return top-K crops ranked by expected revenue.
    
    Parameters:
        state              : str  e.g. "Andhra Pradesh"
        district           : str  e.g. "Anantapur"
        season             : str  e.g. "Kharif"
        land_area_hectares : float e.g. 2.5
        top_k              : int  number of crops to return
    
    Returns:
        DataFrame with ranked crops and revenue estimates
    """
    
    state   = state.strip().title()
    district = district.strip().title()
    season  = season.strip().title()
    
    if state not in le_state.classes_:
        return f"❌ State '{state}' not found. Available: {list(le_state.classes_)}"
    if district not in le_district.classes_:
        return f"❌ District '{district}' not found."
    if season not in le_season.classes_:
        return f"❌ Season '{season}' not found. Available: {list(le_season.classes_)}"
    
    mask = (viable_crops['State_Name']    == state) & \
           (viable_crops['District_Name'] == district) & \
           (viable_crops['Season']        == season)
    
    row = viable_crops[mask]
    if row.empty:
        return f"❌ No crop history found for {district}, {season}"
    
    crops_to_evaluate = list(set(row['Viable_Crops'].values[0]))
    
    crops_to_evaluate = [c for c in crops_to_evaluate if c in le_crop.classes_]
    
    if not crops_to_evaluate:
        return "❌ No matching crops found in model"
    
    rain_row = rainfall_lookup[
        (rainfall_lookup['State_Name']    == state) & 
        (rainfall_lookup['District_Name'] == district)
    ]
    rainfall = rain_row['Annual_Rainfall'].values[0] if not rain_row.empty else df['Annual_Rainfall'].median()
    
    latest_year = int(df['Year'].max())
    
    records = []
    for crop in crops_to_evaluate:
        features = pd.DataFrame([{
            'State_Enc'       : le_state.transform([state])[0],
            'District_Enc'    : le_district.transform([district])[0],
            'Season_Enc'      : le_season.transform([season])[0],
            'Crop_Enc'        : le_crop.transform([crop])[0],
            'Annual_Rainfall' : rainfall,
            'Year'            : latest_year
        }])
        
        predicted_yield = model.predict(features)[0]
        
        price_row = price_lookup[
            (price_lookup['State_Name'] == state) & 
            (price_lookup['Crop']       == crop)
        ]
        
        if price_row.empty:
            continue  
            
        avg_price = price_row['Avg_Price'].values[0]
        
        expected_revenue = predicted_yield * (avg_price * 10) * land_area_hectares
        
        records.append({
            'Crop'              : crop,
            'Predicted_Yield'   : round(predicted_yield, 3),
            'Avg_Price_per_Q'   : round(avg_price, 2),
            'Land_Area_ha'      : land_area_hectares,
            'Expected_Revenue'  : round(expected_revenue, 2)
        })
    
    if not records:
        return "❌ Could not estimate revenue for any crop"
    
    result_df = pd.DataFrame(records)
    result_df = result_df.sort_values('Expected_Revenue', ascending=False)
    result_df = result_df.head(top_k).reset_index(drop=True)
    result_df.index += 1  
    
    return result_df

print("✅ Recommendation function ready")

✅ Recommendation function ready


In [4]:

results = recommend_crops(
    state='Andhra Pradesh',
    district='Anantapur',
    season='Kharif',
    land_area_hectares=2.5,
    top_k=5
)

print(f"\n🌾 Top Crop Recommendations")
print(f"📍 Anantapur, Andhra Pradesh | Season: Kharif | Land: 2.5 ha")
print("=" * 70)
print(results.to_string())


🌾 Top Crop Recommendations
📍 Anantapur, Andhra Pradesh | Season: Kharif | Land: 2.5 ha
           Crop  Predicted_Yield  Avg_Price_per_Q  Land_Area_ha  Expected_Revenue
1        Tomato            3.912         15800.00           2.5        1545170.00
2  Dry Chillies            2.642         18712.62           2.5        1235966.52
3      Turmeric            3.895          9315.50           2.5         907005.36
4        Cotton            1.955          7050.00           2.5         344482.42
5     Groundnut            0.940          6964.60           2.5         163644.29


In [5]:
# Test a few more to validate the system
test_cases = [
    ('Punjab', 'Ludhiana', 'Rabi', 3.0),
    ('Karnataka', 'Mysuru', 'Kharif', 1.5),
    ('Maharashtra', 'Pune', 'Rabi', 2.0),
]

for state, district, season, area in test_cases:
    print(f"\n📍 {district}, {state} | {season} | {area} ha")
    print("-" * 60)
    result = recommend_crops(state, district, season, area)
    if isinstance(result, str):
        print(result)
    else:
        print(result[['Crop', 'Predicted_Yield', 'Expected_Revenue']].to_string())


📍 Ludhiana, Punjab | Rabi | 3.0 ha
------------------------------------------------------------
    Crop  Predicted_Yield  Expected_Revenue
1  Beans            1.438         280423.41

📍 Mysuru, Karnataka | Kharif | 1.5 ha
------------------------------------------------------------
❌ District 'Mysuru' not found.

📍 Pune, Maharashtra | Rabi | 2.0 ha
------------------------------------------------------------
                           Crop  Predicted_Yield  Expected_Revenue
1   Arhar (Tur/Red Gram)(Whole)            0.538         100496.16
2                      Soyabean            1.048          99778.11
3                         Maize            1.925          84160.95
4  Sesamum(Sesame,Gingelly,Til)            0.248          78108.52
5                       Mustard            0.580          73661.46
